In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. Load existing Next-Basket ML predictions
# ============================================================

predictions_df = spark.table(
    "workspace.ml_data.next_basket_lr_validation_predictions"
)

print("Prediction table loaded successfully.")
print("Rows:", predictions_df.count())
print(
    "Customers:",
    predictions_df.select("user_id").distinct().count()
)

display(
    predictions_df
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "purchase_probability",
        "is_new_to_customer",
        "candidate_source",
    )
    .orderBy(
        "user_id",
        F.desc("purchase_probability"),
    )
    .limit(20)
)

Prediction table loaded successfully.
Rows: 2715125
Customers: 26360


user_id,target_order_id,product_id,purchase_probability,is_new_to_customer,candidate_source
14,2316178,29509,0.6871980336715175,0,reorder
14,2316178,23803,0.6460582185396573,0,reorder
14,2316178,8744,0.5080867030927294,0,reorder
14,2316178,37266,0.36486852088734056,0,reorder
14,2316178,15869,0.293682771356237,0,reorder
14,2316178,15172,0.21297882695938164,0,reorder
14,2316178,13966,0.15120584658796776,0,reorder
14,2316178,11418,0.10910158533399805,0,reorder
14,2316178,11131,0.10640666305997226,0,reorder
14,2316178,5077,0.08710147622885822,0,reorder


In [0]:
# ============================================================
# 2. Rank the predicted products for each customer's next basket
# ============================================================

prediction_window = (
    Window
    .partitionBy(
        "user_id",
        "target_order_id",
    )
    .orderBy(
        F.desc("purchase_probability"),
        F.asc("product_id"),
    )
)


top_predictions_df = (
    predictions_df
    .withColumn(
        "prediction_rank",
        F.row_number().over(prediction_window),
    )
    .filter(
        F.col("prediction_rank") <= 10
    )
)


print("Top 10 next-basket predictions created.")
print("Rows:", top_predictions_df.count())
print(
    "Customers:",
    top_predictions_df
    .select("user_id")
    .distinct()
    .count()
)


display(
    top_predictions_df
    .select(
        "user_id",
        "target_order_id",
        "prediction_rank",
        "product_id",
        "purchase_probability",
        "is_new_to_customer",
        "candidate_source",
    )
    .orderBy(
        "user_id",
        "prediction_rank",
    )
    .limit(30)
)

Top 10 next-basket predictions created.
Rows: 263600
Customers: 26360


user_id,target_order_id,prediction_rank,product_id,purchase_probability,is_new_to_customer,candidate_source
14,2316178,1,29509,0.6871980336715175,0,reorder
14,2316178,2,23803,0.6460582185396573,0,reorder
14,2316178,3,8744,0.5080867030927294,0,reorder
14,2316178,4,37266,0.36486852088734056,0,reorder
14,2316178,5,15869,0.293682771356237,0,reorder
14,2316178,6,15172,0.21297882695938164,0,reorder
14,2316178,7,13966,0.15120584658796776,0,reorder
14,2316178,8,11418,0.10910158533399805,0,reorder
14,2316178,9,11131,0.10640666305997226,0,reorder
14,2316178,10,5077,0.08710147622885822,0,reorder


In [0]:
# ============================================================
# 3. Enrich predictions with product information
# ============================================================

products_df = spark.table(
    "workspace.cleaned_data.products"
)

aisles_df = spark.table(
    "workspace.cleaned_data.aisles"
)

departments_df = spark.table(
    "workspace.cleaned_data.departments"
)


top_predictions_enriched_df = (
    top_predictions_df.alias("pred")

    .join(
        products_df.alias("prod"),
        F.col("pred.product_id") == F.col("prod.product_id"),
        "left",
    )

    .join(
        aisles_df.alias("aisle"),
        F.col("prod.aisle_id") == F.col("aisle.aisle_id"),
        "left",
    )

    .join(
        departments_df.alias("dept"),
        F.col("prod.department_id")
        == F.col("dept.department_id"),
        "left",
    )

    .select(
        F.col("pred.user_id"),
        F.col("pred.target_order_id"),
        F.col("pred.prediction_rank"),
        F.col("pred.product_id"),
        F.col("prod.product_name"),
        F.col("aisle.aisle").alias("aisle"),
        F.col("dept.department").alias("department"),
        F.col("pred.purchase_probability"),
        F.col("pred.is_new_to_customer"),
        F.col("pred.candidate_source"),
    )
)


print("Prediction data enriched successfully.")
print("Rows:", top_predictions_enriched_df.count())

display(
    top_predictions_enriched_df
    .orderBy(
        "user_id",
        "prediction_rank",
    )
    .limit(30)
)

Prediction data enriched successfully.
Rows: 263600


user_id,target_order_id,prediction_rank,product_id,product_name,aisle,department,purchase_probability,is_new_to_customer,candidate_source
14,2316178,1,29509,80 Vodka Holiday Edition,spirits,alcohol,0.6871980336715175,0,reorder
14,2316178,2,23803,Jalapeno Pepper,fresh vegetables,produce,0.6460582185396573,0,reorder
14,2316178,3,8744,Mixed Vegetables,frozen produce,frozen,0.5080867030927294,0,reorder
14,2316178,4,37266,Tater Treats Seasoned Shredded Potatoes,frozen appetizers sides,frozen,0.36486852088734056,0,reorder
14,2316178,5,15869,Sweet Hot Dog Buns,buns rolls,bakery,0.293682771356237,0,reorder
14,2316178,6,15172,Seasoned Chicken Fry Batter Mix,marinades meat preparation,pantry,0.21297882695938164,0,reorder
14,2316178,7,13966,Chicken Pot Pie,frozen meals,frozen,0.15120584658796776,0,reorder
14,2316178,8,11418,Malt Vinegar,oils vinegars,pantry,0.10910158533399805,0,reorder
14,2316178,9,11131,Hot Taco Lawry's Spices & Seasonings,marinades meat preparation,pantry,0.10640666305997226,0,reorder
14,2316178,10,5077,100% Whole Wheat Bread,bread,bakery,0.08710147622885822,0,reorder


In [0]:
# ============================================================
# 4. Prepare Next-Basket prediction serving dataset
# ============================================================

next_basket_serving_df = (
    top_predictions_enriched_df

    # Client-friendly prediction type
    .withColumn(
        "prediction_type",
        F.when(
            F.col("is_new_to_customer") == 1,
            F.lit("NEW_DISCOVERY"),
        ).otherwise(
            F.lit("REORDER")
        ),
    )

    # Percentage form for UI display
    .withColumn(
        "predicted_probability_pct",
        F.round(
            F.col("purchase_probability") * 100,
            1,
        ),
    )

    .select(
        "user_id",
        "target_order_id",
        "prediction_rank",
        "product_id",
        "product_name",
        "aisle",
        "department",
        "purchase_probability",
        "predicted_probability_pct",
        "prediction_type",
        "is_new_to_customer",
        "candidate_source",
    )

    .orderBy(
        "user_id",
        "prediction_rank",
    )
)


print("Next-Basket serving dataset prepared.")
print("Rows:", next_basket_serving_df.count())

display(
    next_basket_serving_df.limit(30)
)

Next-Basket serving dataset prepared.
Rows: 263600


user_id,target_order_id,prediction_rank,product_id,product_name,aisle,department,purchase_probability,predicted_probability_pct,prediction_type,is_new_to_customer,candidate_source
14,2316178,1,29509,80 Vodka Holiday Edition,spirits,alcohol,0.6871980336715175,68.7,REORDER,0,reorder
14,2316178,2,23803,Jalapeno Pepper,fresh vegetables,produce,0.6460582185396573,64.6,REORDER,0,reorder
14,2316178,3,8744,Mixed Vegetables,frozen produce,frozen,0.5080867030927294,50.8,REORDER,0,reorder
14,2316178,4,37266,Tater Treats Seasoned Shredded Potatoes,frozen appetizers sides,frozen,0.36486852088734056,36.5,REORDER,0,reorder
14,2316178,5,15869,Sweet Hot Dog Buns,buns rolls,bakery,0.293682771356237,29.4,REORDER,0,reorder
14,2316178,6,15172,Seasoned Chicken Fry Batter Mix,marinades meat preparation,pantry,0.21297882695938164,21.3,REORDER,0,reorder
14,2316178,7,13966,Chicken Pot Pie,frozen meals,frozen,0.15120584658796776,15.1,REORDER,0,reorder
14,2316178,8,11418,Malt Vinegar,oils vinegars,pantry,0.10910158533399805,10.9,REORDER,0,reorder
14,2316178,9,11131,Hot Taco Lawry's Spices & Seasonings,marinades meat preparation,pantry,0.10640666305997226,10.6,REORDER,0,reorder
14,2316178,10,5077,100% Whole Wheat Bread,bread,bakery,0.08710147622885822,8.7,REORDER,0,reorder


In [0]:
# ============================================================
# 5. Validate Next-Basket serving dataset
# ============================================================

row_count = next_basket_serving_df.count()

customer_count = (
    next_basket_serving_df
    .select("user_id")
    .distinct()
    .count()
)

duplicate_count = (
    next_basket_serving_df
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

missing_product_names = (
    next_basket_serving_df
    .filter(
        F.col("product_name").isNull()
    )
    .count()
)

invalid_probabilities = (
    next_basket_serving_df
    .filter(
        (F.col("purchase_probability") < 0)
        |
        (F.col("purchase_probability") > 1)
    )
    .count()
)

invalid_ranks = (
    next_basket_serving_df
    .filter(
        (F.col("prediction_rank") < 1)
        |
        (F.col("prediction_rank") > 10)
    )
    .count()
)


print("NEXT-BASKET SERVING VALIDATION")
print("=" * 45)

print("Rows:", row_count)
print("Customers:", customer_count)
print("Duplicate predictions:", duplicate_count)
print("Missing product names:", missing_product_names)
print("Invalid probabilities:", invalid_probabilities)
print("Invalid prediction ranks:", invalid_ranks)


assert row_count == 263600, (
    f"Unexpected row count: {row_count}"
)

assert customer_count == 26360, (
    f"Unexpected customer count: {customer_count}"
)

assert duplicate_count == 0, (
    f"Duplicate predictions found: {duplicate_count}"
)

assert missing_product_names == 0, (
    f"Missing product names: {missing_product_names}"
)

assert invalid_probabilities == 0, (
    f"Invalid probabilities found: {invalid_probabilities}"
)

assert invalid_ranks == 0, (
    f"Invalid prediction ranks found: {invalid_ranks}"
)


print()
print("Next-Basket prediction validation passed.")

NEXT-BASKET SERVING VALIDATION
Rows: 263600
Customers: 26360
Duplicate predictions: 0
Missing product names: 0
Invalid probabilities: 0
Invalid prediction ranks: 0

Next-Basket prediction validation passed.


In [0]:
# ============================================================
# 6. Neon PostgreSQL connection
# ============================================================

neon_host = (
    "ep-summer-sea-ag9wnzfy-pooler."
    "c-2.eu-central-1.aws.neon.tech"
)

neon_port = "5432"
neon_database = "neondb"
neon_user = "neondb_owner"

# Password remains protected in Databricks Secrets
neon_password = dbutils.secrets.get(
    scope="neon",
    key="password",
)

print("Neon configuration loaded successfully.")
print("Host:", neon_host)
print("Database:", neon_database)
print("User:", neon_user)

Neon configuration loaded successfully.
Host: ep-summer-sea-ag9wnzfy-pooler.c-2.eu-central-1.aws.neon.tech
Database: neondb
User: neondb_owner


In [0]:
# ============================================================
# 7. Export Next-Basket predictions to Neon PostgreSQL
# ============================================================

target_table = "next_basket_predictions"

(
    next_basket_serving_df
    .write
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", target_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .option("batchsize", "5000")
    .option("numPartitions", "2")
    .mode("overwrite")
    .save()
)

print("Export completed:", target_table)
print(
    "Rows exported:",
    next_basket_serving_df.count(),
)

Export completed: next_basket_predictions
Rows exported: 263600


In [0]:
# ============================================================
# 8. Verify Next-Basket predictions stored in Neon
# ============================================================

neon_predictions_df = (
    spark.read
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", target_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .load()
)

neon_row_count = neon_predictions_df.count()

neon_customer_count = (
    neon_predictions_df
    .select("user_id")
    .distinct()
    .count()
)

duplicate_count = (
    neon_predictions_df
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("NEON NEXT-BASKET VALIDATION")
print("=" * 45)

print("Rows:", neon_row_count)
print("Customers:", neon_customer_count)
print("Duplicate predictions:", duplicate_count)

assert neon_row_count == 263600
assert neon_customer_count == 26360
assert duplicate_count == 0

print()
print("Neon Next-Basket validation passed.")

NEON NEXT-BASKET VALIDATION
Rows: 263600
Customers: 26360
Duplicate predictions: 0

Neon Next-Basket validation passed.


In [0]:
display(
    neon_predictions_df
    .filter(F.col("user_id") == 21)
    .orderBy("prediction_rank")
)

user_id,target_order_id,prediction_rank,product_id,product_name,aisle,department,purchase_probability,predicted_probability_pct,prediction_type,is_new_to_customer,candidate_source
21,1854765,1,23729,Hard Boiled Eggs,eggs,dairy eggs,0.5683535426858906,56.8,REORDER,0,reorder
21,1854765,2,44632,Sparkling Water Grapefruit,water seltzer sparkling water,beverages,0.3617915822651947,36.2,REORDER,0,reorder
21,1854765,3,28204,Organic Fuji Apple,fresh fruits,produce,0.33565992748081386,33.6,REORDER,0,reorder
21,1854765,4,48988,Unsweetened Premium Iced Tea,tea,beverages,0.27522307917062994,27.5,REORDER,0,reorder
21,1854765,5,33894,Goldfish Cheddar Baked Snack Crackers Multi Packs,crackers,snacks,0.24767552364038758,24.8,REORDER,0,reorder
21,1854765,6,18523,Total 2% All Natural Greek Strained Yogurt with Honey,yogurt,dairy eggs,0.19697456412499348,19.7,REORDER,0,reorder
21,1854765,7,31387,Natural Almonds 100 Calorie Packs,nuts seeds dried fruit,snacks,0.15123152567067855,15.1,REORDER,0,reorder
21,1854765,8,27548,Original Semisoft Cheese,packaged cheese,dairy eggs,0.13706932981336928,13.7,REORDER,0,reorder
21,1854765,9,49235,Organic Half & Half,cream,dairy eggs,0.13051068017228473,13.1,REORDER,0,reorder
21,1854765,10,14788,Unsweetened Iced Tea,tea,beverages,0.12525956500966695,12.5,REORDER,0,reorder
